# Jevlet on Colab: daily-driver training and Jev-style ablations

Two tracks share one data pipeline:

* **Daily driver (Jevlet-P)** — a pretrained encoder under the packed System-One topology
  (shared state, isolated question branches, option-boundary pointer head). Trained here,
  exported as a slim fp16 checkpoint, and run on the laptop by `scripts.desktop --checkpoint`.
* **Research** — one-axis-at-a-time ablations (topology, pooling, head, proper-scoring loss,
  backbone) through the resumable successive-halving runner.

**Runtime.** Any GPU works; **L4** or **A100** gives BF16 and is preferred. The notebook uses
the PyTorch that Colab ships and **never installs CUDA drivers** — the preinstalled wheel is
built against the VM's driver, and mismatched installs are the usual cause of broken
runtimes. Everything important is synced to Drive or a *private* Hugging Face repo, so a
disconnect costs at most a few minutes: rerun all cells with the same settings to resume.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

REPO_URL = ""  # e.g. https://github.com/<you>/Jevlet.git once pushed
DRIVE_SOURCE_FOLDER = "/content/drive/MyDrive/Jevlet-source"  # used when REPO_URL is empty
PERSIST_TO = "drive"  # "drive" or "hf"
HF_REPO_ID = ""  # private model repo, required only for "hf"

PUBLIC_DATASETS = ["banking77", "boolq", "mnli", "massive", "clinc"]
PUBLIC_LIMITS = dict(train_limit=20_000, dev_limit=1_500, vault_limit=1_500)
DAILY_COUNTS = (60_000, 4_000, 2_000)
SEED = 1337

TRAIN_DAILY_DRIVER = True
DAILY_STEPS = 12_000
DAILY_BACKBONE = "sentence-transformers/all-MiniLM-L6-v2"  # laptop CPU friendly (~40 ms/call)

RUN_RESEARCH = True
RESEARCH_HOURS = 4.0

print("torch", torch.__version__, "| CUDA build", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("Runtime > Change runtime type > GPU, then rerun")
name = torch.cuda.get_device_name(0)
print("GPU:", name, "| capability", torch.cuda.get_device_capability(0))
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"])
BF16 = torch.cuda.is_bf16_supported()
AMP = "bf16" if BF16 else "fp16"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
BIG_GPU = torch.cuda.get_device_properties(0).total_memory > 20 * 2**30
BATCH = 32 if BIG_GPU else 16
print("mixed precision:", AMP, "| batch", BATCH)

## Source, dependencies, persistence
The working copy lives on `/content` for speed; only checkpoints, run JSON, logs, and exports
go to Drive/HF. No token is stored in the notebook — HF login prompts interactively.

In [ ]:
if PERSIST_TO == "drive" or not REPO_URL:
    from google.colab import drive

    drive.mount("/content/drive")
PROJECT = Path("/content/Jevlet")
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)
    else:
        if not Path(DRIVE_SOURCE_FOLDER).is_dir():
            raise FileNotFoundError("Set REPO_URL or copy the repo to DRIVE_SOURCE_FOLDER")
        shutil.copytree(
            DRIVE_SOURCE_FOLDER, PROJECT, ignore=shutil.ignore_patterns("data", "results")
        )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJECT}[semantic,colab]"], check=True
)
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)  # scripts resolve configs and the model cache relative to the repo

from notebooks.colab_runtime import CheckpointSync, restore_checkpoint, restore_tree  # noqa: E402

RUNS = Path("/content/jevlet_runs")
RUNS.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT = Path("/content/drive/MyDrive/Jevlet/checkpoints") if PERSIST_TO == "drive" else None
if PERSIST_TO == "hf":
    if not HF_REPO_ID:
        raise ValueError("Set HF_REPO_ID to a private model repository")
    from huggingface_hub import HfApi, notebook_login

    notebook_login()
    HfApi().create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)
PERSIST = dict(drive_root=DRIVE_ROOT, hf_repo_id=HF_REPO_ID or None)

## Data: every source, pinned
Public sources are pinned to exact Hub revisions (locked on first run and reused on resume),
then combined with the synthetic System-One families and the daily-driver route/risk
generator into one hash-pinned mixture. Vault rows are written separately and never read by
training or research. The hand-written daily benchmark (`jevlet/benchmarks.py`) is not part
of any mixture.

In [ ]:
from jevlet.daily_synthetic import generate_daily_dataset
from jevlet.mixture import build_mixture
from jevlet.public_data import download_public_data
from jevlet.synthetic import generate_dataset

DATA = Path("/content/jevlet_data")
stored = restore_checkpoint("dataset_manifest.json", RUNS, **PERSIST)
locked = json.loads(stored.read_text()) if stored else None
revisions = {k: v["revision"] for k, v in locked["public"]["sources"].items()} if locked else None
public = download_public_data(
    DATA / "public", PUBLIC_DATASETS, seed=SEED, revisions=revisions, **PUBLIC_LIMITS
)
generate_dataset(DATA / "synthetic", count=30_000, seed=SEED)
generate_daily_dataset(DATA / "daily", counts=DAILY_COUNTS, seed=2026)
mixture = build_mixture(
    {"synthetic": DATA / "synthetic", "public": DATA / "public", "daily": DATA / "daily"},
    DATA / "mixture",
    seed=SEED,
)
if locked and locked["mixture"]["train"]["sha256"] != mixture["splits"]["train"]["sha256"]:
    raise RuntimeError("mixture hash changed since the stored checkpoints; refusing to resume")
(RUNS / "dataset_manifest.json").write_text(
    json.dumps({"public": public, "mixture": mixture["splits"]}, indent=2)
)
CheckpointSync(RUNS, **PERSIST).sync_once()
print(json.dumps({split: info["rows"] for split, info in mixture["splits"].items()}))

## Daily driver: train Jevlet-P
Resumes from the synced `last.pt` if one exists. Metrics are on the mixture dev split;
the daily benchmark below is the transfer test that decides whether to use it.

In [ ]:
from jevlet.training import train_experiment

DAILY_RUN = RUNS / "daily_driver"
config = json.loads((PROJECT / "configs/pretrained_daily.json").read_text())
config["model"]["backbone"] = DAILY_BACKBONE
config["data"].update(
    train=str(DATA / "mixture/train.jsonl"),
    dev=str(DATA / "mixture/dev.jsonl"),
    vault=str(DATA / "mixture/vault/vault.jsonl"),
)
config["training"].update(
    batch_size=BATCH,
    gradient_accumulation=1,
    max_steps=DAILY_STEPS,
    save_every_steps=500,
    warmup_steps=400,
    amp_dtype=AMP,
)
config.pop("gpu_memory_fraction", None)
if TRAIN_DAILY_DRIVER:
    resumed = restore_checkpoint("daily_driver/last.pt", RUNS, **PERSIST)
    if resumed:
        config["training"]["resume_from"] = str(resumed)
        print("resuming from", resumed)
    sync = CheckpointSync(RUNS, interval_seconds=240, **PERSIST)
    sync.start()
    try:
        metrics = train_experiment(config, DAILY_RUN)
    finally:
        sync.stop()
    print(
        {
            k: metrics[k]
            for k in (
                "accuracy",
                "ece",
                "calibrated_ece",
                "option_order_prediction_agreement",
                "steps",
            )
        }
    )

In [ ]:
# Transfer test on hand-written laptop tasks, then a slim fp16 export for the laptop.
subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.eval_daily",
        "--checkpoint",
        str(DAILY_RUN / "best.pt"),
        "--output",
        str(DAILY_RUN / "daily_benchmark.json"),
    ],
    check=True,
)
(RUNS / "export").mkdir(exist_ok=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.export_checkpoint",
        str(DAILY_RUN / "best.pt"),
        str(RUNS / "export" / "jevlet-p-daily.pt"),
        "--fp16",
    ],
    check=True,
)
CheckpointSync(RUNS, **PERSIST).sync_once()

## Research: Jev-style ablations (resumable)
Nine hypotheses, one axis each, promoted by Pareto layers over quality, calibration,
robustness, throughput, latency, and VRAM. Stage results and `last.pt` files sync off the VM;
rerunning this cell after a disconnect skips finished candidates and continues the rest.

In [ ]:
from research.runner import run_successive_halving

RESEARCH = RUNS / "research"
if RUN_RESEARCH:
    restored = restore_tree("research", RUNS, **PERSIST)
    print("restored research files:", restored)
    research = json.loads((PROJECT / "configs/overnight_pretrained.json").read_text())
    research["data"].update(train=config["data"]["train"], dev=config["data"]["dev"])
    research["training"].update(batch_size=BATCH, gradient_accumulation=1, amp_dtype=AMP)
    research.pop("gpu_memory_fraction", None)
    research["research"].update(hours=RESEARCH_HOURS, log=str(RESEARCH / "experiment_log.jsonl"))
    sync = CheckpointSync(RUNS, interval_seconds=300, **PERSIST)
    sync.start()
    try:
        summary = run_successive_halving(research, RESEARCH)
    finally:
        sync.stop()
    print(json.dumps(summary.get("winner", summary), indent=2)[:3000])

## Bring the daily driver home
On the laptop (from `D:\Jevlet`), copy `export/jevlet-p-daily.pt` from Drive
(`MyDrive/Jevlet/checkpoints/export/`) or the private HF repo to `data\daily\current.pt`, then:

```powershell
python -m scripts.eval_daily --checkpoint data\daily\current.pt
python -m scripts.desktop --checkpoint data\daily\current.pt
```

Keep the zero-shot router if the checkpoint does not beat it on the daily benchmark. Colab
sessions can end without warning even on paid plans; confirm the export exists in Drive/HF
before closing the tab.